Install Dependencies

In [1]:
!pip install wandb ptflops --quiet

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms

import numpy as np
import wandb

from ptflops import get_model_complexity_info


Custom CIFAR-10 Dataset & DataLoader

In [3]:
class CIFAR10Dataset(Dataset):
    def __init__(self, train=True):
        self.transform = transforms.Compose([
            transforms.RandomHorizontalFlip() if train else transforms.Lambda(lambda x: x),
            transforms.ToTensor(),
            transforms.Normalize(
                (0.4914, 0.4822, 0.4465),
                (0.247, 0.243, 0.261)
            )
        ])

        self.dataset = datasets.CIFAR10(
            root="./data",
            train=train,
            download=True,
            transform=self.transform
        )

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        return self.dataset[idx]


def get_dataloader(batch_size=128, train=True):
    return DataLoader(
        CIFAR10Dataset(train=train),
        batch_size=batch_size,
        shuffle=train,
        num_workers=2,
        pin_memory=True
    )


CNN Model

In [4]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(64 * 8 * 8, 256)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))

        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)


FLOPs Calculation

In [5]:
model = CNN()

macs, params = get_model_complexity_info(
    model,
    (3, 32, 32),
    as_strings=True,
    print_per_layer_stat=False
)

print("FLOPs (MACs):", macs)
print("Parameters:", params)


FLOPs (MACs): 6.85 MMac
Parameters: 1.07 M


Gradient & Weight Flow Visualization

In [6]:
def log_gradient_flow(model, step):
    layers, grads = [], []

    for name, param in model.named_parameters():
        if param.requires_grad and param.grad is not None:
            layers.append(name)
            grads.append(param.grad.abs().mean().item())

    wandb.log({
        "Gradient Flow": wandb.plot.bar(
            wandb.Table(
                data=[[l, g] for l, g in zip(layers, grads)],
                columns=["Layer", "Mean Gradient"]
            ),
            "Layer",
            "Mean Gradient"
        )
    }, step=step)


def log_weight_flow(model, step):
    layers, weights = [], []

    for name, param in model.named_parameters():
        layers.append(name)
        weights.append(param.data.abs().mean().item())

    wandb.log({
        "Weight Flow": wandb.plot.bar(
            wandb.Table(
                data=[[l, w] for l, w in zip(layers, weights)],
                columns=["Layer", "Mean Weight"]
            ),
            "Layer",
            "Mean Weight"
        )
    }, step=step)


Training Loop + Wandb

In [7]:
EPOCHS = 30
BATCH_SIZE = 128
LR = 0.001

device = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = device == "cuda"

wandb.init(
    project="cifar10-lab2-amp",
    config={
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "lr": LR,
        "use_amp": USE_AMP
    }
)

train_loader = get_dataloader(BATCH_SIZE, train=True)

model = CNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

step = 0

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()

        log_gradient_flow(model, step)
        log_weight_flow(model, step)

        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()
        step += 1

    wandb.log({
        "epoch": epoch,
        "train_loss": epoch_loss / len(train_loader)
    })

    print(f"Epoch [{epoch+1}/{EPOCHS}] "
          f"Loss: {epoch_loss/len(train_loader):.4f}")

wandb.finish()


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 1


wandb: You chose 'Create a W&B account'
wandb: Create an account here: https://wandb.ai/authorize?signup=true&ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: m25csa001 (m25csa001-indian-institute-of-technology-jodhpur) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


100%|██████████| 170M/170M [00:02<00:00, 77.7MB/s]
/tmp/ipython-input-3354395796.py:24: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
/tmp/ipython-input-3354395796.py:38: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):
/tmp/ipython-input-3354395796.py:38: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


Epoch [1/30] Loss: 1.3311
Epoch [2/30] Loss: 0.9599
Epoch [3/30] Loss: 0.8185
Epoch [4/30] Loss: 0.7243
Epoch [5/30] Loss: 0.6466
Epoch [6/30] Loss: 0.5784
Epoch [7/30] Loss: 0.5151
Epoch [8/30] Loss: 0.4598
Epoch [9/30] Loss: 0.4122
Epoch [10/30] Loss: 0.3595
Epoch [11/30] Loss: 0.3131
Epoch [12/30] Loss: 0.2843
Epoch [13/30] Loss: 0.2396
Epoch [14/30] Loss: 0.2159
Epoch [15/30] Loss: 0.1867
Epoch [16/30] Loss: 0.1626
Epoch [17/30] Loss: 0.1465
Epoch [18/30] Loss: 0.1282
Epoch [19/30] Loss: 0.1113
Epoch [20/30] Loss: 0.1051
Epoch [21/30] Loss: 0.0932
Epoch [22/30] Loss: 0.0884
Epoch [23/30] Loss: 0.0808
Epoch [24/30] Loss: 0.0743
Epoch [25/30] Loss: 0.0746
Epoch [26/30] Loss: 0.0714
Epoch [27/30] Loss: 0.0593
Epoch [28/30] Loss: 0.0672
Epoch [29/30] Loss: 0.0634
Epoch [30/30] Loss: 0.0628


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train_loss,█▆▅▅▄▄▄▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,29
train_loss,0.06282
